# LendingClub dashboard — data preparation and validation

## tl;dr

This notebook reads the supplied LendingClub CSV without changing it, validates the one-loan-per-row grain, confirms the 151-column source schema, profiles only the selected analytical columns, fixes numeric/date typing in Python, and produces the reviewed aggregate cube used by the interactive dashboard.

The final outcome and loss metrics follow the requested policy: only `Fully Paid`, `Default`, and `Charged Off` enter the matured denominator; `Current`, `Late`, and `In Grace Period` are excluded. Expected loss, LGD, EAD, recovery, and contribution remain explicitly labeled proxies because exposure at default and operating/funding costs are not directly supplied.

## Context & Methods

### Key assumptions

- Grain: one accepted LendingClub loan per row; `id` is the candidate loan key.
- Default outcome: `loan_status` is exactly `Default` or `Charged Off`.
- Final outcome population: `Fully Paid`, `Default`, or `Charged Off` only.
- Performance windows: a defaulted loan is counted at the month of its last payment; a loan enters the 12/24/36-month denominator only if it was issued at least that many months before the data cutoff (latest `last_pymnt_d`).
- Vintage: calendar year of `issue_d`.
- EAD proxy: original `funded_amnt`, because balance at default is unavailable.
- Default-exposure proxy for LGD/recovery: `max(funded_amnt - total_rec_prncp, 0)` on defaulted/charged-off loans.
- Net-loss proxy: `max(default-exposure proxy - recoveries, 0)`.
- LGD proxy: aggregate net-loss proxy / aggregate default-exposure proxy.
- Expected-loss-rate proxy: matured PD × LGD proxy; expected loss = rate × funded amount.
- Funding and operating costs are not in the source. They are adjustable scenario assumptions in the dashboard, not observed values.
- Post-origination fields are retained only for realized-performance/proxy calculations and are marked as leakage for origination-time modeling.

In [1]:
from pathlib import Path
from collections import Counter
import json
import math
import numpy as np
import pandas as pd

SOURCE = Path.cwd() / "data" / "raw" / "accepted_2010_to_2017.csv"
PROJECT = Path.cwd() / "outputs" / "lendingclub_dashboard"

DATA_DIR = PROJECT / "data"
EVIDENCE_DIR = PROJECT / "evidence"

DATA_DIR.mkdir(parents=True, exist_ok=True)
EVIDENCE_DIR.mkdir(parents=True, exist_ok=True)

CHUNK_SIZE = 25_000
FINAL_STATUSES = {"Fully Paid", "Default", "Charged Off"}
DEFAULT_STATUSES = {"Default", "Charged Off"}
SMALL_SAMPLE_THRESHOLD = 100

SOURCE, SOURCE.stat().st_size

(WindowsPath('data/raw/accepted_2010_to_2017.csv'),
 1293034426)

## Data

### 1. Profile and aggregate the supplied CSV

In [2]:
selected_columns = [
    "id", "loan_amnt", "funded_amnt", "term", "int_rate", "installment",
    "grade", "sub_grade", "home_ownership", "annual_inc", "issue_d",
    "loan_status", "purpose", "dti", "fico_range_low", "fico_range_high",
    "application_type", "total_pymnt", "total_pymnt_inv", "total_rec_prncp",
    "total_rec_int", "recoveries", "collection_recovery_fee", "last_pymnt_d",
    "hardship_flag", "debt_settlement_flag",
]
numeric_columns = [
    "loan_amnt", "funded_amnt", "int_rate", "installment", "annual_inc", "dti",
    "fico_range_low", "fico_range_high", "total_pymnt", "total_pymnt_inv",
    "total_rec_prncp", "total_rec_int", "recoveries", "collection_recovery_fee",
]
dimensions = [
    "vintage_year", "grade", "term", "purpose", "fico_band", "dti_band",
    "application_type", "home_ownership", "income_band",
]
metric_columns = [
    "loan_count", "funded_amount", "loan_amount", "funded_x_apr",
    "matured_count", "default_count", "charged_off_count", "fully_paid_count",
    "default_exposure_proxy", "recoveries_default", "net_loss_proxy",
    "realized_interest", "principal_repaid",
    "seasoned_12m_count", "default_12m_count", "seasoned_24m_count", "default_24m_count",
    "seasoned_36m_count", "default_36m_count",
    "current_count", "late_count", "prepaid_count", "prepaid_24m_count", "settlement_count", "collection_fee_default",
    "matured_funded", "matured_funded_x_apr", "matured_interest",
]

PERFORMANCE_WINDOWS = (12, 24, 36)
LATE_STATUSES = {"In Grace Period", "Late (16-30 days)", "Late (31-120 days)"}
PREPAY_MONTHS_EARLY = 3

# Data cutoff = latest payment month in the file; needed to know which loans are old enough for each window.
last_payment_months = pd.to_datetime(
    pd.read_csv(SOURCE, usecols=["last_pymnt_d"])["last_pymnt_d"], format="%b-%Y", errors="coerce"
)
DATA_CUTOFF = last_payment_months.max()
del last_payment_months

header = pd.read_csv(SOURCE, nrows=0).columns.tolist()
missing_counts = pd.Series(0, index=selected_columns, dtype="int64")
status_counts = Counter()
id_arrays = []
cube_parts = []
row_count = 0
invalid_issue_dates = 0
numeric_parse_failures = Counter()

fico_bins = [-np.inf, 660, 680, 700, 720, 740, np.inf]
fico_labels = ["<660", "660–679", "680–699", "700–719", "720–739", "740+"]
dti_bins = [-np.inf, 10, 20, 30, 40, np.inf]
dti_labels = ["<10", "10–19.99", "20–29.99", "30–39.99", "40+"]
income_bins = [-np.inf, 40_000, 60_000, 80_000, 100_000, 150_000, np.inf]
income_labels = ["<40K", "40–59K", "60–79K", "80–99K", "100–149K", "150K+"]

for chunk_index, chunk in enumerate(pd.read_csv(SOURCE, usecols=selected_columns, chunksize=CHUNK_SIZE, low_memory=False)):
    row_count += len(chunk)
    missing_counts = missing_counts.add(chunk.isna().sum(), fill_value=0).astype("int64")
    work = chunk.copy()

    ids = pd.to_numeric(work["id"], errors="coerce")
    id_arrays.append(ids.to_numpy(dtype="float64", copy=True))
    status_counts.update(work["loan_status"].fillna("<Missing>").astype(str))

    for col in numeric_columns:
        original_non_null = work[col].notna()
        parsed = pd.to_numeric(work[col], errors="coerce")
        numeric_parse_failures[col] += int((original_non_null & parsed.isna()).sum())
        work[col] = parsed

    issue_date = pd.to_datetime(work["issue_d"], format="%b-%Y", errors="coerce")
    invalid_issue_dates += int((work["issue_d"].notna() & issue_date.isna()).sum())
    work["vintage_year"] = issue_date.dt.year.astype("Int64").astype(str).replace("<NA>", "Unknown")

    for col in ["grade", "term", "purpose", "application_type", "home_ownership"]:
        work[col] = work[col].fillna("Unknown").astype(str).str.strip()
    work["grade"] = work["grade"].str.upper()
    work["home_ownership"] = work["home_ownership"].replace({"ANY": "OTHER", "NONE": "OTHER"})

    fico_mid = (work["fico_range_low"] + work["fico_range_high"]) / 2
    work["fico_band"] = pd.cut(fico_mid, fico_bins, labels=fico_labels, right=False).astype("object").fillna("Unknown")
    work["dti_band"] = pd.cut(work["dti"], dti_bins, labels=dti_labels, right=False).astype("object").fillna("Unknown")
    work["income_band"] = pd.cut(work["annual_inc"], income_bins, labels=income_labels, right=False).astype("object").fillna("Unknown")

    status = work["loan_status"].fillna("<Missing>").astype(str)
    matured = status.isin(FINAL_STATUSES)
    defaulted = status.isin(DEFAULT_STATUSES)
    charged_off = status.eq("Charged Off")

    funded = work["funded_amnt"].fillna(0).clip(lower=0)
    principal_repaid = work["total_rec_prncp"].fillna(0).clip(lower=0)
    recoveries = work["recoveries"].fillna(0).clip(lower=0)
    default_exposure = (funded - principal_repaid).clip(lower=0).where(defaulted, 0)
    recoveries_default = recoveries.where(defaulted, 0)
    net_loss = (default_exposure - recoveries_default).clip(lower=0)

    work["loan_count"] = 1
    work["funded_amount"] = funded
    work["loan_amount"] = work["loan_amnt"].fillna(0).clip(lower=0)
    work["funded_x_apr"] = funded * work["int_rate"].fillna(0) / 100.0
    work["matured_count"] = matured.astype("int64")
    work["default_count"] = defaulted.astype("int64")
    work["charged_off_count"] = charged_off.astype("int64")
    work["fully_paid_count"] = status.eq("Fully Paid").astype("int64")
    work["default_exposure_proxy"] = default_exposure
    work["recoveries_default"] = recoveries_default
    work["net_loss_proxy"] = net_loss
    work["realized_interest"] = work["total_rec_int"].fillna(0).clip(lower=0)
    work["principal_repaid"] = principal_repaid

    # Timing: months on book at the last payment (default proxy = last payment month) and at the data cutoff.
    term_months = pd.to_numeric(work["term"].str.extract(r"(\d+)")[0], errors="coerce")
    last_payment = pd.to_datetime(work["last_pymnt_d"], format="%b-%Y", errors="coerce")
    mob_last = ((last_payment.dt.year - issue_date.dt.year) * 12 + (last_payment.dt.month - issue_date.dt.month)).fillna(0)
    mob_cutoff = (DATA_CUTOFF.year - issue_date.dt.year) * 12 + (DATA_CUTOFF.month - issue_date.dt.month)
    for window in PERFORMANCE_WINDOWS:
        seasoned = mob_cutoff >= window
        work[f"seasoned_{window}m_count"] = seasoned.astype("int64")
        work[f"default_{window}m_count"] = (seasoned & defaulted & (mob_last <= window)).astype("int64")

    work["current_count"] = status.eq("Current").astype("int64")
    work["late_count"] = status.isin(LATE_STATUSES).astype("int64")
    work["prepaid_count"] = (status.eq("Fully Paid") & (mob_last <= term_months - PREPAY_MONTHS_EARLY)).astype("int64")
    work["prepaid_24m_count"] = (work["prepaid_count"].astype(bool) & (mob_last <= 24) & work["seasoned_24m_count"].astype(bool)).astype("int64")
    work["settlement_count"] = (defaulted & work["debt_settlement_flag"].astype(str).str.strip().eq("Y")).astype("int64")
    work["collection_fee_default"] = work["collection_recovery_fee"].fillna(0).clip(lower=0).where(defaulted, 0)
    work["matured_funded"] = funded.where(matured, 0)
    work["matured_funded_x_apr"] = work["funded_x_apr"].where(matured, 0)
    work["matured_interest"] = work["realized_interest"].where(matured, 0)

    grouped = work.groupby(dimensions, dropna=False, observed=True)[metric_columns].sum().reset_index()
    cube_parts.append(grouped)

    if (chunk_index + 1) % 10 == 0:
        print(f"Processed {row_count:,} rows")

cube = pd.concat(cube_parts, ignore_index=True)
cube = cube.groupby(dimensions, dropna=False, observed=True)[metric_columns].sum().reset_index()
cube = cube.sort_values(dimensions, kind="stable").reset_index(drop=True)

all_ids = pd.Series(np.concatenate(id_arrays), name="id")
missing_id_count = int(all_ids.isna().sum())
duplicate_loan_id_rows = int(all_ids.dropna().duplicated(keep=False).sum())
duplicate_loan_id_values = int(all_ids.dropna()[all_ids.dropna().duplicated(keep=False)].nunique())

missing_report = pd.DataFrame({
    "column": selected_columns,
    "missing_count": [int(missing_counts[c]) for c in selected_columns],
})
missing_report["missing_pct"] = missing_report["missing_count"] / row_count
missing_report = missing_report.sort_values(["missing_pct", "column"], ascending=[False, True]).reset_index(drop=True)

status_report = pd.DataFrame(status_counts.items(), columns=["loan_status", "row_count"]).sort_values("row_count", ascending=False)

print({
    "rows": row_count,
    "columns": len(header),
    "cube_rows": len(cube),
    "missing_id_count": missing_id_count,
    "duplicate_loan_id_rows": duplicate_loan_id_rows,
    "duplicate_loan_id_values": duplicate_loan_id_values,
    "invalid_issue_dates": invalid_issue_dates,
    "data_cutoff": str(DATA_CUTOFF.date()),
})
status_report

Processed 250,000 rows


Processed 500,000 rows


Processed 750,000 rows


Processed 1,000,000 rows


Processed 1,250,000 rows


Processed 1,500,000 rows


Processed 1,750,000 rows


{'rows': 1757149, 'columns': 151, 'cube_rows': 132733, 'missing_id_count': 0, 'duplicate_loan_id_rows': 0, 'duplicate_loan_id_values': 0, 'invalid_issue_dates': 0, 'data_cutoff': '2019-03-01'}


,loan_status,row_count
0,Fully Paid,1023664
1,Current,451136
2,Charged Off,258806
4,Late (31-120 days),14246
3,In Grace Period,5517
5,Late (16-30 days),2746
7,Does not meet the credit policy. Status:Fully ...,731
8,Does not meet the credit policy. Status:Charge...,270
6,Default,33


## Results

### 2. Reconcile core metrics and save reviewed evidence

In [3]:
def safe_div(num, den):
    return float(num / den) if den and not pd.isna(den) else None

totals = cube[metric_columns].sum()
pd_rate = safe_div(totals["default_count"], totals["matured_count"])
charged_off_rate = safe_div(totals["charged_off_count"], totals["matured_count"])
lgd_proxy = safe_div(totals["net_loss_proxy"], totals["default_exposure_proxy"])
recovery_rate_proxy = safe_div(totals["recoveries_default"], totals["default_exposure_proxy"])
weighted_apr = safe_div(totals["funded_x_apr"], totals["funded_amount"])
expected_loss_rate_proxy = pd_rate * lgd_proxy if pd_rate is not None and lgd_proxy is not None else None
expected_loss_proxy = totals["funded_amount"] * expected_loss_rate_proxy if expected_loss_rate_proxy is not None else None

core_kpis = {
    "total_loans": int(totals["loan_count"]),
    "total_funded_amount": float(totals["funded_amount"]),
    "average_funded_amount": safe_div(totals["funded_amount"], totals["loan_count"]),
    "matured_loans": int(totals["matured_count"]),
    "default_loans": int(totals["default_count"]),
    "charged_off_loans": int(totals["charged_off_count"]),
    "matured_default_rate": pd_rate,
    "charged_off_rate": charged_off_rate,
    "weighted_apr": weighted_apr,
    "default_exposure_proxy": float(totals["default_exposure_proxy"]),
    "lgd_proxy": lgd_proxy,
    "recovery_rate_proxy": recovery_rate_proxy,
    "expected_loss_rate_proxy": expected_loss_rate_proxy,
    "expected_loss_proxy": float(expected_loss_proxy) if expected_loss_proxy is not None else None,
}

qa_summary = {
    "source_file": str(SOURCE),
    "source_bytes": SOURCE.stat().st_size,
    "row_count": row_count,
    "column_count": len(header),
    "selected_column_count": len(selected_columns),
    "selected_columns": selected_columns,
    "grain": "One accepted LendingClub loan per row; candidate key id.",
    "missing_id_count": missing_id_count,
    "duplicate_loan_id_rows": duplicate_loan_id_rows,
    "duplicate_loan_id_values": duplicate_loan_id_values,
    "invalid_issue_dates": invalid_issue_dates,
    "numeric_parse_failures": dict(numeric_parse_failures),
    "loan_status_values": status_report.to_dict(orient="records"),
    "cube_row_count": len(cube),
    "data_cutoff": str(DATA_CUTOFF.date()),
    "small_sample_threshold_matured": SMALL_SAMPLE_THRESHOLD,
    "core_kpis": core_kpis,
}

missing_report.to_csv(EVIDENCE_DIR / "missing_values_selected_columns.csv", index=False, encoding="utf-8-sig")
status_report.to_csv(EVIDENCE_DIR / "loan_status_counts.csv", index=False, encoding="utf-8-sig")
cube.to_csv(DATA_DIR / "lendingclub_segment_cube.csv", index=False, encoding="utf-8")
(EVIDENCE_DIR / "qa_summary.json").write_text(json.dumps(qa_summary, indent=2, ensure_ascii=False), encoding="utf-8")
(EVIDENCE_DIR / "core_kpis.json").write_text(json.dumps(core_kpis, indent=2, ensure_ascii=False), encoding="utf-8")

core_kpis

{'total_loans': 1757149,
 'total_funded_amount': 25991364125.0,
 'average_funded_amount': 14791.781530763754,
 'matured_loans': 1282503,
 'default_loans': 258839,
 'charged_off_loans': 258806,
 'matured_default_rate': 0.2018233095751043,
 'charged_off_rate': 0.2017975786411416,
 'weighted_apr': 0.13580036838639767,
 'default_exposure_proxy': 2853937690.4800005,
 'lgd_proxy': 0.8882540732918517,
 'recovery_rate_proxy': 0.11189995616540664,
 'expected_loss_rate_proxy': 0.17927037681532876,
 'expected_loss_proxy': 4659481640.633167}

### 3. Create the dashboard snapshot with explicit provenance and proxy definitions

In [4]:
cube_records = json.loads(cube.to_json(orient="records", double_precision=10))
generated_at = pd.Timestamp.now(tz="UTC").isoformat()

metric_definitions = [
    {"label":"Default rate within 12/24/36 months","definition":"Defaulted loans whose last payment month is within N months of issue, divided by loans issued at least N months before the data cutoff. Comparable across vintages because every loan is measured over the same age.","formula":"default_Nm_count / seasoned_Nm_count","dependencies":["default_12m_count","seasoned_12m_count","default_24m_count","seasoned_24m_count","default_36m_count","seasoned_36m_count"],"componentIds":["vintage-quality"]},
    {"label":"Current delinquency rate","definition":"Loans In Grace Period or Late divided by active loans (Current + late). Early-warning signal for young vintages.","formula":"late_count / (current_count + late_count)","dependencies":["late_count","current_count"],"componentIds":["vintage-quality"]},
    {"label":"Prepayment rate within 24 months","definition":"Loans fully paid at least 3 months before term end and within 24 months of issue, divided by loans issued at least 24 months before the data cutoff. prepaid_count / fully_paid_count is not comparable across vintages because young vintages' Fully Paid loans are early payoffs by construction.","formula":"prepaid_24m_count / seasoned_24m_count","dependencies":["prepaid_24m_count","seasoned_24m_count"],"componentIds":["vintage-quality"]},
    {"label":"Matured contribution margin","definition":"Contribution computed only on loans with a final outcome. For young vintages the closed loans are dominated by early defaults, so this is a pessimistic bound; the all-loan contribution margin is the optimistic bound. They converge as a vintage matures.","formula":"(matured_interest - net_loss_proxy - cost_rate * matured balance-years) / matured_funded","dependencies":["matured_interest","net_loss_proxy","matured_funded","matured_funded_x_apr"],"componentIds":["vintage-quality"]},
    {"label":"Total loans","definition":"Count of accepted-loan rows after applying dashboard filters.","componentIds":["kpi-total-loans","vintage-table","action-table"]},
    {"label":"Funded amount","definition":"Sum of funded_amnt after applying dashboard filters.","componentIds":["kpi-funded","vintage-table","action-table"]},
    {"label":"Matured default rate","definition":"Loans with loan_status equal to Default or Charged Off divided by loans whose status is Fully Paid, Default, or Charged Off. Current, Late, and In Grace Period are excluded from the denominator.","formula":"default_count / matured_count","dependencies":["default_count","matured_count"],"componentIds":["kpi-pd","vintage-risk","risk-purpose","risk-heatmap-fico-dti","risk-heatmap-grade-term","action-table"]},
    {"label":"Charged-off rate","definition":"Charged Off loans divided by loans whose status is Fully Paid, Default, or Charged Off.","formula":"charged_off_count / matured_count","dependencies":["charged_off_count","matured_count"],"componentIds":["kpi-chargeoff"]},
    {"label":"Weighted APR","definition":"Funded-amount-weighted average of int_rate.","formula":"sum(funded_amnt * int_rate) / sum(funded_amnt)","dependencies":["funded_x_apr","funded_amount"],"componentIds":["kpi-apr","vintage-table","grade-apr-el"]},
    {"label":"LGD proxy","definition":"Net principal loss proxy divided by default-exposure proxy for Default and Charged Off loans. Default exposure is approximated as max(funded_amnt - total_rec_prncp, 0); net loss subtracts recoveries. This is not observed exposure at default.","formula":"net_loss_proxy / default_exposure_proxy","dependencies":["net_loss_proxy","default_exposure_proxy"],"componentIds":["kpi-lgd","vintage-table","action-table"]},
    {"label":"Recovery-rate proxy","definition":"Recoveries divided by the default-exposure proxy for Default and Charged Off loans. This is not a contractual recovery rate.","formula":"recoveries_default / default_exposure_proxy","dependencies":["recoveries_default","default_exposure_proxy"],"componentIds":["kpi-recovery"]},
    {"label":"Expected-loss proxy","definition":"Matured default rate multiplied by LGD proxy and filtered funded amount used as an EAD proxy. Original funded amount is not exposure at default.","formula":"PD * LGD_proxy * funded_amount","dependencies":["default_count","matured_count","net_loss_proxy","default_exposure_proxy","funded_amount"],"componentIds":["kpi-el","vintage-table","action-table"]},
    {"label":"Risk-adjusted contribution-rate proxy","definition":"Weighted APR minus expected-loss-rate proxy minus adjustable funding-cost and operating-cost assumptions. It excludes timing, prepayment, servicing revenue, taxes, capital and other costs.","formula":"weighted_apr - expected_loss_rate_proxy - funding_cost_assumption - operating_cost_assumption","componentIds":["kpi-contribution","contribution-bridge","action-table"]},
]

snapshot = {
    "surface": "dashboard",
    "title": "LendingClub Portfolio & Credit Risk",
    "generatedAt": generated_at,
    "status": "reviewed",
    "buildStatus": "creating",
    "exportPreferences": {"selected": None, "destinations": []},
    "filters": [
        {"id":"vintage_year","label":"Năm giải ngân","field":"vintage_year","defaultValue":"all"},
        {"id":"grade","label":"Grade","field":"grade","defaultValue":"all"},
        {"id":"term","label":"Term","field":"term","defaultValue":"all"},
        {"id":"purpose","label":"Purpose","field":"purpose","defaultValue":"all"},
        {"id":"fico_band","label":"FICO band","field":"fico_band","defaultValue":"all"},
        {"id":"dti_band","label":"DTI band","field":"dti_band","defaultValue":"all"},
        {"id":"application_type","label":"Application type","field":"application_type","defaultValue":"all"},
        {"id":"home_ownership","label":"Home ownership","field":"home_ownership","defaultValue":"all"},
    ],
    "queries": {
        "portfolio_cube": {
            "rows": cube_records,
            "source": {
                "label": "Supplied LendingClub accepted-loan CSV, reviewed aggregate cube",
                "files": [str(SOURCE)],
                "filters": ["Issue vintages present in source: 2010–2017", "Final-outcome denominator: Fully Paid, Default, Charged Off only"],
                "executedAt": generated_at,
                "coverage": {"startDate":"2010-01-01","endDate":"2017-12-31","timezone":"Source month values; no intraday timezone"},
                "evidenceFlow": [
                    {"title":"Source","detail":f"Read {SOURCE.name} directly; original file unchanged."},
                    {"title":"Validation","detail":f"{row_count:,} rows, {len(header)} columns; duplicate loan-id rows: {duplicate_loan_id_rows:,}; invalid issue dates: {invalid_issue_dates:,}."},
                    {"title":"Transformation","detail":"Chunked pandas preparation; decimal fields parsed from raw CSV text, categorical bands derived, then additive metrics aggregated to dashboard filter grain."},
                ],
                "metricDefinitions": metric_definitions,
                "caveats": [
                    "Accepted loans only; approval/rejection analysis is not supported.",
                    "Current, Late, and In Grace Period are excluded from matured default-rate denominators.",
                    "LGD, EAD, expected loss, recovery, and contribution are proxies under documented assumptions.",
                    "Post-origination fields used in performance proxies would leak future information in an origination-time model.",
                ],
            },
        },
        "data_quality": {
            "rows": json.loads(missing_report.to_json(orient="records", double_precision=10)),
            "source": {
                "label": "Missing-value profile across selected analytical columns",
                "files": [str(SOURCE)],
                "executedAt": generated_at,
                "metricDefinitions": [{"label":"Missing percentage","definition":"Missing cells divided by total source rows for each column.","formula":"missing_count / row_count","componentIds":["method-data-quality"]}],
            },
        },
        "loan_status_profile": {
            "rows": status_report.to_dict(orient="records"),
            "source": {"label":"Observed loan_status values in supplied CSV","files":[str(SOURCE)],"executedAt":generated_at},
        },
    },
    "qa": qa_summary,
}

snapshot_path = DATA_DIR / "dashboard_snapshot.json"
snapshot_path.write_text(json.dumps(snapshot, ensure_ascii=False, separators=(",", ":")), encoding="utf-8")
print({"snapshot": str(snapshot_path), "snapshot_mb": round(snapshot_path.stat().st_size / 1024**2, 2), "cube_rows": len(cube)})

{'snapshot': 'outputs\\lendingclub_dashboard\\data\\dashboard_snapshot.json', 'snapshot_mb': 107.6, 'cube_rows': 132733}


## Takeaways

- Use the executed QA outputs in `evidence/` to confirm grain, missingness, status values, and key uniqueness.
- Use `data/dashboard_snapshot.json` as the only dashboard data snapshot; it contains reviewed aggregates and explicit provenance.
- Treat all LGD/EAD/expected-loss/recovery/contribution outputs as proxies, not source-reported actuals.
- Do not use post-origination fields as predictors in an origination-time model.